# Specimen 03 — Embeddings & Semantic Search

Goal: turn text into vectors and search by meaning instead of exact keyword match. Hand-roll the search once before Specimen 04 lets a library do it for you.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')


## 1. Pick a small document set

20-50 short text chunks of your own choosing — notes, articles, anything you have on hand.

In [ ]:
documents = [
    "The Wright brothers achieved the first powered flight in 1903 near Kitty Hawk, North Carolina.",
    "Python is a high-level programming language known for its readability and broad standard library.",
    "The mitochondria is the organelle responsible for producing ATP, the energy currency of the cell.",
    "Photosynthesis converts light energy into chemical energy stored in glucose.",
    "The stock market crash of 1929 triggered the Great Depression in the United States.",
    "Neural networks are loosely inspired by the structure of biological neurons in the brain.",
    "The Great Wall of China was built over centuries to protect against invasions from the north.",
    "Machine learning models improve their performance by learning patterns from training data.",
    "The French Revolution began in 1789 and led to the end of the monarchy in France.",
    "DNA carries the genetic instructions used in the growth and functioning of living organisms.",
    "Climate change is driven largely by the accumulation of greenhouse gases in the atmosphere.",
    "The Roman Empire at its height stretched from Britain to the Middle East.",
    "Quantum computers use qubits, which can exist in superposition, unlike classical bits.",
    "The human heart pumps blood through a network of arteries, veins, and capillaries.",
    "Shakespeare wrote 37 plays and over 150 sonnets during the late 16th and early 17th centuries.",
    "Renewable energy sources like solar and wind are becoming cheaper than fossil fuels in many regions.",
    "The Amazon rainforest produces roughly 20% of the world's oxygen and is called Earth's lungs.",
    "Blockchain is a distributed ledger technology that underlies cryptocurrencies like Bitcoin.",
    "Volcanic eruptions occur when magma, gases, and ash escape from below Earth's crust.",
    "The Apollo 11 mission landed the first humans on the Moon in July 1969.",
    "Antibiotics work by killing bacteria or stopping their growth, but are ineffective against viruses.",
    "The Industrial Revolution transformed manufacturing from hand production to machines.",
    "Coral reefs support roughly 25% of all marine species despite covering less than 1% of the ocean floor.",
    "GDP measures the total monetary value of goods and services produced within a country.",
    "The printing press, invented by Gutenberg around 1440, revolutionized the spread of information.",
]

print(f'{len(documents)} document chunks loaded')

## 2. Embed every chunk

Local model, no API key needed. Turn each chunk into a vector.

In [ ]:
doc_embeddings = embedder.encode(documents, convert_to_numpy=True)
print('Embedding shape:', doc_embeddings.shape)

## 3. Embed a query

Same model, same vector space.

In [ ]:
def embed_query(query):
    return embedder.encode([query], convert_to_numpy=True)[0]

sample_query = "How did humans first reach the Moon?"
query_vec = embed_query(sample_query)
print('Query vector shape:', query_vec.shape)

## 4. Hand-roll cosine similarity

Compute it yourself with plain numpy against every chunk — don't call a library's `.search()` for this first pass.

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def semantic_search(query, top_k=3):
    q_vec = embed_query(query)
    scores = [cosine_similarity(q_vec, doc_vec) for doc_vec in doc_embeddings]
    ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

for doc, sim in semantic_search(sample_query):
    print(f'{sim:.3f}  {doc}')

## 5. Return top-k results

Sort by similarity, print the top 3-5 chunks for a few different test queries.

In [ ]:
test_queries = [
    "How did humans first reach the Moon?",
    "What causes plants to grow using sunlight?",
    "How do computers that use qubits work?",
    "Why are coral reefs important?",
]

for query in test_queries:
    print(f'\nQuery: {query}')
    for doc, sim in semantic_search(query, top_k=3):
        print(f'  {sim:.3f}  {doc}')

## 6. Compare against keyword search

Same queries, plain substring match. Where does semantic search actually win, and where does it lose to keyword search?

In [ ]:
def keyword_search(query, top_k=3):
    query_words = set(query.lower().split())
    scores = []
    for doc in documents:
        doc_words = set(doc.lower().replace('.', '').replace(',', '').split())
        overlap = len(query_words & doc_words)
        scores.append(overlap)
    ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

comparison_queries = [
    "Apollo 11 mission",
    "How did people first travel to Earth's natural satellite?",
]

for query in comparison_queries:
    print(f'\nQuery: {query}')
    print('  Semantic:')
    for doc, sim in semantic_search(query, top_k=2):
        print(f'    {sim:.3f}  {doc}')
    print('  Keyword:')
    for doc, overlap in keyword_search(query, top_k=2):
        print(f'    {overlap} shared words  {doc}')